# VCC 2026 · 2 — random-effects meta-analysis

For each target × gene, average the backgrounds that measured it (one vote per background: T cells = CD4_Stim48hr only).
The prediction is the precision-weighted average (Model 2): each background weighted by 1/(SE² + τ²), with τ² per gene from DerSimonian–Laird pooled over all panel targets. No extra shrinking.
The knocked-down gene uses the mean of the sources' on-target LFC. Finishing: >5 CPM gate, DE call (|Δ|/SE_pred > z* and |Δ| > min size), global α; z*, min size and α are tuned on the exam (H1 held out) with cell-eval2.

## After submission: results, lessons and ideas
Recorded 2026-10-14. The follow-up notebook `vcc_2026_2a` (Oct 12–14) was folded into this summary and deleted; its edits live on in `vcc_2026_3`, `3a` and `3b`.

**Leaderboard (A–C, scaled; 0 = the line's true generic knockdown response, 1 = a real replicate)**

| Submission | overall | pds | mse | nmae | fidelity | reach | Jaccard |
|---|---|---|---|---|---|---|---|
| v1 (`vcc_2026`) | −0.270 | 0.088 | 0 | 0.023 | −1.688 | 0.029 | −0.074 |
| **v2 (this notebook, gated `re_003`: z* = 3, min 0.1, α = 2)** | **−0.054** | 0.350 | 0 | 0.024 | −0.805 | 0.083 | 0.026 |
| v3 (`vcc_2026_3`: top 2,000 per knockdown + detection floor) | +0.091 | 0.416 | 0 | 0.048 | −0.088 | 0.192 | −0.024 |
| v3a (`vcc_2026_3a`: top 1,000 + floor) | +0.087 | 0.420 | 0 | 0.076 | −0.121 | 0.180 | −0.032 |

**Lessons**
1. **Choose settings with the leaderboard formula, not average rank.** Re-scored with cell-eval2's generic baseline and replicate anchor, the submitted `re_003` shows the leaderboard's own pattern (fidelity < 0, mse 0, pds > 0), and z* = 2 would have won (H1 overall 0.064 vs 0.009). Average rank had discarded it. Every later notebook selects on the scaled overall.
2. **The gate called far too few genes.** About 29 genes per knockdown against ~1,900 real DE genes on H1. Calling the top N by |Δ| / SE instead (N = 2,000) took the leaderboard from −0.054 to +0.091, the biggest gain so far.
3. **Called genes must be detectable.** Each called gene is raised to the smallest change a 400-cell test against the line's controls detects at z = 5: 92–94% of called genes then come out significant in the generated cells (48–72% without; 80% at z = 4).
4. **H1 ranks models like the leaderboard but misjudges sizes.** It ranked v1 < v2 < v3 and 1,000 < 2,000 correctly, but forecast +0.079 for v2 → v3 (actual +0.145) and 0.014 for 2,000 vs 1,000 (actual 0.004). H1 also penalises the floor-raised effects (nmae) and rewards extra calls (Jaccard) more than A–C do.
5. **Fewer calls do not fix Jaccard.** v3a halved N: only nmae improved; fidelity, reach and Jaccard fell. The weak point is which genes are called, not how many: on H1, 19% of the top 2,000 are real DE genes against 18% for random genes, and the direction is right 62% of the time.
6. **Source agreement:** the unanimous sign rule (a gene in ≥ 2 lines only if they all agree) lowered H1 from 0.118 to 0.106 (bootstrap 95% interval −0.019 to −0.005), and keeping only genes measured in ≥ 2 lines gave 0.104 (−0.037 to +0.008). One-line genes are the weakest calls (15% significant, 61% right), but dropping them did not pay.
7. **Corrected values on Model 2's genes** (`vcc_2026_3b`: Model 2 picks the top 2,000, each takes the ridge-corrected `corrected_k5` value): 0.119 against 0.118, inside the noise (95% interval −0.004 to +0.007; ahead in 70% of resamples). Bootstrapping the 75 targets puts H1's noise between arms at about ±0.01.
8. **Generic response (D5 / D6):** H1's shared knockdown response is in the span of its own control PCs (gene-CV R² 0.35; K562 0.42) but no rule from controls transfers between lines (H1 → K562 r = −0.05). Lines agree with each other only weakly (leave-one-out r: HCT116 0.53, HEK293T 0.51 among cancer lines; H1 0.01). The generic-plus-specific split was not built.
9. **mse stays at 0** for every arm: generated cells are more dispersed than real ones (even "no change" scores 1.147 against a bar of 0.953).
10. **Verify long jobs after launching them.** The 50-target `2a` tuning was stopped for submission 3, then crashed on relaunch (a non-empty output folder) and was reported as running when it was not.

**Ideas not yet tested**
- **Context-aware weighting** in `combine`: weight each source by how similar its controls are to the challenge line, w = s(line, background) / (SE² + τ²), with SE_pred = √(Σ w² (SE² + τ²)) / Σ w. τ² and `fit` are unchanged; the A / B / C tables would then differ for the first time.
- **α for top-N calling** (1 / 2 / 3): every top-N arm scored so far used α = 1.
- **K562 as a second validation line** (hold K562 out of `fit`; its raw cells are in the bucket): choose settings that hold on both lines.
- **Generic-response probe:** add an average knockdown response (H1's, or HCT116 / HEK293T / K562's) to every knockdown at full size, never DE-called. Only a submission can test it.
- **Per-knockdown N** scaled by predicted strength, and the generator noise that keeps mse at 0.

## 0. Session
Paths, the `run` / `bg` / `status` helpers, and input checks.

In [ ]:
import os, sys, shutil, subprocess, shlex, json
from pathlib import Path
import numpy as np
import pandas as pd
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V / 'code' / 'vcc_2026_2'), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_2'),
           B='gs://vcc2026', PY=sys.executable, WORK=str(V / 'lib' / 'vcc_2026_2'), SOURCES=str(V / 'lib' / 'sources'),
           SCRIPTS=str(V / 'scripts'))          # PY = this notebook's Python (your venv)
os.environ.update(ENV)
for d in ('CODE', 'LOGS', 'LIB', 'DATA', 'WORK'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now (from the code folder), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/vcc_2026_2/<name>.log."""
    session = 'vcc2_' + name
    if subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0:
        raise RuntimeError(f'{session} is already running; wait for completion')
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    session = 'vcc2_' + name
    alive = subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else 'STOPPED - read the log below')
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

# ---- checks ----
need = ['data/gene_names.csv', 'data/pert_counts.csv', 'data/context_A.h5ad', 'data/context_B.h5ad', 'data/context_C.h5ad',
        'lib/h1/H1_full_2025.h5ad', 'scripts/knockdown_filters.py', 'scripts/build_panel_sources.py']
missing = [f for f in need if not (V / f).exists()]
print('python:', sys.executable)
print('data files: all present' if not missing else f'MISSING input files: {missing}')
print('tmux:', 'OK' if shutil.which('tmux') else 'MISSING -> in the terminal: sudo apt-get install -y tmux')
for m in ['numpy', 'pandas', 'scipy', 'h5py', 'anndata']:
    try:
        __import__(m)
    except ImportError:
        print(f'package {m} MISSING -> in this cell: %pip install {m}')
print('official evaluator:', shutil.which('cell-eval2') or 'MISSING: pip install -r requirements-eval.txt')
H1_CELLS = V / 'lib/h1/H1_full_2025.h5ad'  # raw H1 cells: the exam's real side
LINES = ['A', 'B', 'C']                    # use D/E/F with the final bundle
RUN_NAME = 'vcc_2026_2_meta'
ENV['H1_CELLS'] = str(H1_CELLS); os.environ['H1_CELLS'] = str(H1_CELLS)

Optional: download the raw inputs from the bucket.

In [ ]:
DOWNLOAD_INPUTS = False
if DOWNLOAD_INPUTS:
    run(r"""
    mkdir -p "$LIB/orion" "$LIB/tcell" "$LIB/h1"
    gcloud storage cp --no-clobber "$B/data/*" "$DATA/"
    gcloud storage cp --no-clobber "$B/orion/*" "$LIB/orion/"
    gcloud storage cp --no-clobber "$B/tcell/*" "$LIB/tcell/"
    gcloud storage cp --no-clobber "$B/h1/*" "$LIB/h1/"
    """)

## 1. Scripts
Run all three cells after editing; they write to `~/vcc/code/vcc_2026_2/`.

### vcc2_common.py
Shared helpers: seeds, JSON, the >5 CPM gate, count checks, control moments.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_common.py
"""Shared helpers for the VCC 2026 meta-analysis pipeline.

Imported by:
  vcc2_meta.py      : read_names, save_json
  vcc2_generate.py  : seed_for, save_json, expression_gate, int64_indptr, validate_counts, alignment, moments
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

CONTROL = "non-targeting"


def read_names(path, col):
    """One column of a CSV as a numpy string array (first column if `col` is absent)."""
    df = pd.read_csv(path)
    return np.asarray(df[col] if col in df.columns else df.iloc[:, 0], dtype=str)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=1, default=_json_default))


def seed_for(seed, *keys):
    """Deterministic seed from (seed, keys...): stable across runs and machines (unlike Python's hash())."""
    h = hashlib.sha256(repr((int(seed),) + tuple(str(k) for k in keys)).encode()).digest()
    return int.from_bytes(h[:8], "little")


def expression_gate(mean_cpm):
    """Strict scorer boundary, using arithmetic mean per-cell CPM in real controls."""
    mean = np.asarray(mean_cpm, dtype=float)
    return np.isfinite(mean) & (mean > 5.0)


def int64_indptr(matrix):
    """CSR with an int64 row pointer.

    An h5ad X that is written once and then grown with append() keeps the dtype of its
    FIRST block's indptr. scipy uses int32 for small blocks, which caps the finished file at
    2**31 stored values (merged H1 has ~3.5e9; a 360k-cell submission is ~2.2e9).
    """
    m = matrix.tocsr()
    m.indptr = m.indptr.astype(np.int64)
    return m


def validate_counts(matrix, label="counts"):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix)
    if not (np.isfinite(values).all() and (values >= 0).all() and (values == np.floor(values)).all()):
        raise ValueError(f"{label}: expected finite, nonnegative integer raw counts")


def alignment(source_names, genes):
    """For each of OUR genes, its column in the source (-1 = the source did not measure it)."""
    pos = {g: i for i, g in enumerate(np.asarray(source_names, dtype=str))}
    idx = np.array([pos.get(g, -1) for g in np.asarray(genes, dtype=str)], dtype=np.int64)
    if (idx >= 0).sum() == 0:
        raise ValueError("no genes in common with the provided gene list")
    return idx


def _matrix(a, layer):
    if layer is None:
        return a.X
    if layer in a.layers:
        return a.layers[layer]
    raise KeyError(f"layer {layer!r} not found")


def moments(a, rows, idx, layer=None, block=512):
    """Per-gene statistics over `rows` (cells), streamed `block` cells at a time. Library size = sum over OUR genes.
    Returns arrays aligned to our gene list (NaN where the source lacks the gene)."""
    X = _matrix(a, layer); G = len(idx); ok = idx >= 0; cols = idx[ok]
    s_cnt = np.zeros(ok.sum()); s_cpm = np.zeros(ok.sum()); s_cpm2 = np.zeros(ok.sum()); s_det = np.zeros(ok.sum())
    libs = []
    for b in range(0, len(rows), block):
        r = np.asarray(rows[b:b + block])
        M = X[r]
        M = (M.tocsr() if sp.issparse(M) else sp.csr_matrix(np.asarray(M)))[:, cols].astype(np.float64)
        validate_counts(M)
        lib = np.asarray(M.sum(1)).ravel(); libs.append(lib)
        C = sp.diags(1e6 / np.maximum(lib, 1)) @ M
        s_cnt += np.asarray(M.sum(0)).ravel(); s_cpm += np.asarray(C.sum(0)).ravel()
        s_cpm2 += np.asarray(C.multiply(C).sum(0)).ravel(); s_det += np.asarray((M > 0).sum(0)).ravel()
    n = len(rows); lib = np.concatenate(libs) if libs else np.array([0.0])

    def full(v):
        out = np.full(G, np.nan, dtype=np.float32); out[ok] = v; return out
    if n == 0:
        raise ValueError("no cells to profile")
    mean = s_cpm / n; var = np.maximum(s_cpm2 / n - mean ** 2, 0)
    pooled = s_cnt / max(s_cnt.sum(), 1) * 1e6
    stats = dict(mean_cpm=np.where(ok, 0.0, np.nan), pooled_cpm=full(pooled), bulk_cpm=full(pooled), log_mean=full(np.log1p(pooled)),
                 detection=full(s_det / n), var_cpm=full(var), se_mean=full(np.sqrt(var / n)),
                 n_cells=np.int64(n), median_library=np.float64(np.median(lib)), measured=ok.copy(),
                 legacy_baseline=np.bool_(False))
    stats["mean_cpm"][ok] = mean  # float64 preserves the strict 5-CPM boundary
    return stats

### vcc2_meta.py
Steps 1–5: one vote per background, per-gene τ² (DerSimonian–Laird), precision-weighted average with its SE, on-target LFC.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_meta.py
"""Random-effects meta-analysis of the public panel sources: precision-weighted average (Model 2).

Inputs: lib/sources/<NAME>/effects_panel.h5ad from scripts/build_panel_sources.py (575 targets x 18,533 genes,
layers lfc / se, NaN = not available). One source per background: the T cells enter as CD4_Stim48hr only.

  fit      per-gene tau2: DerSimonian-Laird, pooled over every target with >= 2 backgrounds -> model dir
  predict  one delta table per line, in the folder format vcc2_generate.py reads:
             re.npy, re_se.npy         weight = 1/(SE^2 + tau2); prediction = sum(weight x lfc) / sum(weight);
                                       SE = sqrt(1 / sum(weight))
             n_backgrounds.npy         backgrounds behind each value; 0 = no information (delta stored as NaN)
             ontarget_lfc.npy          mean of the sources' ontarget_lfc: the knocked-down gene, applied separately
  exam     the held-out background's kept targets that at least one other background also kept
  explain  one target x gene: each background's lfc, SE, weight and share, then the prediction

The knocked-down gene's own column never enters tau2 or the averages. No shrinkage: the average is the prediction.

Example: python vcc2_meta.py fit --sources-dir ~/vcc/lib/sources --genes gene_names.csv --holdout H1 --out models/exam
         python vcc2_meta.py predict --model models/exam --targets exam_targets.csv --lines H1 --out exam
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from vcc2_common import read_names, save_json

SCHEMA = 'vcc_2026_2_meta_v1'
SOURCES = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr', 'H1']   # step 1: Stim48hr is the one T-cell vote
MODELS = ('re',)


def background_of(root, name):
    a = ad.read_h5ad(Path(root) / name / 'effects_panel.h5ad', backed='r')
    try: return str(a.uns['background']), a.obs.copy()
    finally: a.file.close()


def included(root, names, holdout=None):
    """Sources that vote, in order; refuses two sources from one background."""
    chosen, seen = [], {}
    for name in names:
        background, _ = background_of(root, name)
        if background == holdout: continue
        if background in seen: raise ValueError(f'{name} and {seen[background]} are both {background}: one vote per background')
        seen[background] = name; chosen.append((name, background))
    if not chosen: raise ValueError('no source backgrounds remain')
    return chosen


class Panel:
    """Included sources stacked as sources x targets x genes (lfc, se), plus each source's on-target LFC."""
    def __init__(self, root, sources, genes):
        self.names = [n for n, _ in sources]; self.backgrounds = [b for _, b in sources]
        lfc, se, own = [], [], []
        for name in self.names:
            a = ad.read_h5ad(Path(root) / name / 'effects_panel.h5ad')
            if not np.array_equal(a.var_names.astype(str), genes): raise ValueError(f'{name}: gene order differs from gene_names.csv')
            targets = a.obs_names.astype(str).to_numpy()
            if lfc and not np.array_equal(targets, self.targets): raise ValueError(f'{name}: target order differs between panels')
            self.targets = targets
            kept = (a.obs['status'] == 'kept').to_numpy()
            lfc.append(np.asarray(a.layers['lfc'], np.float32)); se.append(np.asarray(a.layers['se'], np.float32))
            own.append(np.where(kept, a.obs['ontarget_lfc'].to_numpy(float), np.nan))
        self.lfc, self.se, self.ontarget = np.stack(lfc), np.stack(se), np.stack(own)
        gp = {g: i for i, g in enumerate(genes)}
        self.own_col = np.array([gp.get(t, -1) for t in self.targets])
        self.row = {t: i for i, t in enumerate(self.targets)}


def tau2_dl(lfc, se, own_col):
    """Per-gene between-background variance, DerSimonian-Laird pooled over targets.

    For each target with k >= 2 backgrounds: Q = sum w (y - mu_FE)^2 with w = 1/SE^2 (noise alone predicts k - 1)
    and C = sum w - sum w^2 / sum w.  tau2 = max(0, (sum Q - sum (k - 1)) / sum C), floored at 0.
    Returns tau2 (NaN where no target ever had two backgrounds) and the pooled degrees of freedom.
    """
    _, T, G = lfc.shape
    Q, df, C = np.zeros(G), np.zeros(G), np.zeros(G)
    for t in range(T):
        y, v = lfc[:, t].astype(np.float64), se[:, t].astype(np.float64) ** 2
        ok = np.isfinite(y) & np.isfinite(v) & (v > 0)
        if own_col[t] >= 0: ok[:, own_col[t]] = False
        k = ok.sum(0); use = k >= 2
        if not use.any(): continue
        w = np.where(ok, 1 / np.where(ok, v, 1), 0); y = np.where(ok, y, 0)
        sw = w.sum(0); mu = np.divide((w * y).sum(0), sw, out=np.zeros(G), where=sw > 0)
        Q[use] += (w * (y - mu) ** 2).sum(0)[use]; df[use] += k[use] - 1
        C[use] += (sw - np.divide((w ** 2).sum(0), sw, out=np.zeros(G), where=sw > 0))[use]
    tau2 = np.full(G, np.nan); est = C > 0
    tau2[est] = np.maximum(0, (Q[est] - df[est]) / C[est])
    return tau2, df


def combine(lfc, se, tau2):
    """Sources x rows x genes -> (re, re_se, k) per row x gene. k == 0 gives NaN (no information)."""
    y = lfc.astype(np.float64); total = se.astype(np.float64) ** 2 + tau2   # SE^2 + tau2
    ok = np.isfinite(y) & np.isfinite(total)
    k = ok.sum(0); y = np.where(ok, y, 0)
    w = np.where(ok, 1 / np.where(ok, total, 1), 0); sw = w.sum(0)
    with np.errstate(divide='ignore', invalid='ignore'):
        out = ((w * y).sum(0) / sw, np.sqrt(1 / sw))
    return tuple(np.where(k > 0, x, np.nan).astype(np.float32) for x in out) + (k.astype(np.uint8),)


def fit(sources_dir, genes_csv, output, holdout=None, names=SOURCES):
    genes = read_names(genes_csv, 'gene_name'); sources = included(sources_dir, names, holdout)
    panel = Panel(sources_dir, sources, genes)
    tau2, df = tau2_dl(panel.lfc, panel.se, panel.own_col)
    est = np.isfinite(tau2); fallback = float(np.median(tau2[est]))
    measured_in = np.isfinite(panel.lfc).any(1).sum(0)                  # backgrounds measuring each gene at all
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    np.save(out / 'genes.npy', genes); np.save(out / 'tau2.npy', tau2.astype(np.float32)); np.save(out / 'tau2_df.npy', df)
    q = np.quantile(tau2[est], [0.1, 0.25, 0.5, 0.75, 0.9])
    model = dict(schema=SCHEMA, sources_dir=str(Path(sources_dir).resolve()), genes_csv=str(Path(genes_csv).resolve()),
                 sources=[n for n, _ in sources], backgrounds=[b for _, b in sources], heldout_background=holdout,
                 tau2_estimator='DerSimonian-Laird, pooled over targets with >= 2 backgrounds, own column excluded, floored at 0',
                 tau2_fallback=fallback, tau2_fallback_rule='median tau2 for genes no target measured in 2 backgrounds',
                 genes_estimated=int(est.sum()), genes_fallback=int((~est & (measured_in > 0)).sum()),
                 genes_no_data=int((measured_in == 0).sum()), genes_tau2_zero=int((tau2[est] == 0).sum()),
                 tau2_quantiles=dict(zip(['10%', '25%', '50%', '75%', '90%'], q.round(5).tolist())))
    save_json(out / 'model.json', model)
    print(f'fit: {len(sources)} backgrounds {model["backgrounds"]} (held out: {holdout})')
    print(f'  tau2 estimated for {model["genes_estimated"]:,} genes ({model["genes_tau2_zero"]:,} at 0), '
          f'median {fallback:.4f}; {model["genes_fallback"]:,} genes measured in one background use the median; '
          f'{model["genes_no_data"]:,} genes have no data anywhere')
    return model


def load_model(model_dir):
    m = Path(model_dir); model = json.loads((m / 'model.json').read_text())
    if model.get('schema') != SCHEMA: raise ValueError('incompatible model; fit with this notebook')
    genes = np.load(m / 'genes.npy').astype(str); tau2 = np.load(m / 'tau2.npy').astype(np.float64)
    used = np.where(np.isfinite(tau2), tau2, model['tau2_fallback'])
    panel = Panel(model['sources_dir'], list(zip(model['sources'], model['backgrounds'])), genes)
    return model, genes, used, panel


def predict(model_dir, targets_csv, lines, output, chunk=64):
    model, genes, tau2, panel = load_model(model_dir)
    targets = read_names(targets_csv, 'target_gene')
    if not len(targets): raise ValueError('empty perturbation panel')
    out = Path(output)
    for line in lines:   # an earlier meta prediction is replaced (it is deterministic); anything else is kept
        d = out / line; earlier = d / 'prediction_report.json'
        if d.exists() and any(d.iterdir()) and not (earlier.exists() and json.loads(earlier.read_text()).get('schema') == SCHEMA):
            raise ValueError(f'choose an empty prediction directory: {d}')
        if d.exists():
            for old in [*d.glob('*.npy'), d / 'prediction_report.json']: old.unlink(missing_ok=True)
    T, G = len(targets), len(genes)
    arrays = {name: np.full((T, G), np.nan, np.float32) for name in ['re', 're_se']}
    count = np.zeros((T, G), np.uint8); ontarget = np.full(T, np.nan, np.float32)
    rows = np.array([panel.row.get(t, -1) for t in targets]); missing = targets[rows < 0].tolist()
    gp = {g: i for i, g in enumerate(genes)}
    for start in range(0, T, chunk):
        idx = np.arange(start, min(start + chunk, T)); idx = idx[rows[idx] >= 0]
        if not len(idx): continue
        lfc, se = panel.lfc[:, rows[idx]].copy(), panel.se[:, rows[idx]]
        for j, i in enumerate(idx):
            if targets[i] in gp: lfc[:, j, gp[targets[i]]] = np.nan   # step 5: own gene is not averaged
        arrays['re'][idx], arrays['re_se'][idx], k = combine(lfc, se, tau2)
        count[idx] = k
        own = panel.ontarget[:, rows[idx]]; n = np.isfinite(own).sum(0)
        ontarget[idx] = np.divide(np.nan_to_num(own).sum(0), n, out=np.full(len(idx), np.nan), where=n > 0)
    voters = [[b for b, kept in zip(panel.backgrounds, np.isfinite(panel.ontarget[:, r])) if kept] if r >= 0 else [] for r in rows]
    info = (count > 0).sum(1)
    report = dict(schema=SCHEMA, models=list(MODELS), heldout_background=model['heldout_background'],
                  source_backgrounds=model['backgrounds'], source_names=model['sources'], model_dir=str(Path(model_dir).resolve()),
                  targets=str(Path(targets_csv).resolve()), weighting='1 / (SE^2 + tau2)', shrinkage='none', ontarget='mean of the backgrounds\' ontarget_lfc, kept rows only',
                  no_information='NaN in re (n_backgrounds 0); the generator predicts no change there',
                  not_in_panel=missing, no_source_targets=[t for t, v in zip(targets.tolist(), voters) if not v],
                  coverage=[dict(target=t, backgrounds=v, genes_with_information=int(n)) for t, v, n in zip(targets.tolist(), voters, info)])
    for line in lines:
        d = out / line; d.mkdir(parents=True, exist_ok=True)
        np.save(d / 'genes.npy', genes); np.save(d / 'targets.npy', targets)
        for name, value in arrays.items(): np.save(d / f'{name}.npy', value)
        np.save(d / 'n_backgrounds.npy', count); np.save(d / 'ontarget_lfc.npy', ontarget)
        save_json(d / 'prediction_report.json', dict(report, line=line))
    by_k = pd.Series([len(v) for v in voters]).value_counts().sort_index()
    print(f'predict: {T} targets x {G:,} genes -> {", ".join(str(out / l) for l in lines)}')
    print('  targets by number of backgrounds: ' + ', '.join(f'{k}: {n}' for k, n in by_k.items()))
    print(f'  no source at all (all genes "no information"): {report["no_source_targets"] or "none"}')
    if missing: print(f'  not in the panel: {missing}')
    return report


def exam(sources_dir, holdout, output, names=SOURCES):
    status = {}
    for name in names:
        background, obs = background_of(sources_dir, name)
        status.setdefault(background, []).append(obs['status'].eq('kept'))
    if holdout not in status: raise ValueError(f'no source for {holdout}')
    held = pd.concat(status.pop(holdout), axis=1).any(axis=1)
    others = pd.concat([pd.concat(v, axis=1).any(axis=1) for v in status.values()], axis=1).sum(axis=1)
    targets = held.index[held & (others >= 1)].tolist()
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'target_gene': targets}).to_csv(output, index=False)
    print(f'exam: {len(targets)} {holdout} targets also kept in >= 1 other background -> {output}')
    return targets


def explain(model_dir, target, gene):
    """The worked example for one target x gene, from the real panel."""
    if target == gene: raise ValueError('the knocked-down gene is not averaged; see ontarget_lfc')
    model, genes, tau2, panel = load_model(model_dir)
    r, g = panel.row[target], int(np.flatnonzero(genes == gene)[0])
    t2 = tau2[g]; y = panel.lfc[:, r, g].astype(float); se = panel.se[:, r, g].astype(float)
    frame = pd.DataFrame(dict(background=panel.backgrounds, lfc=y, se=se)).set_index('background')
    frame['se2_plus_tau2'] = se ** 2 + t2; frame['weight'] = 1 / frame.se2_plus_tau2
    frame['share'] = frame.weight / frame.weight.sum()
    re, re_se, k = (x[0, 0] for x in combine(frame.lfc.to_numpy()[:, None, None],
                                                                 frame.se.to_numpy()[:, None, None], np.array([t2])))
    summary = dict(target=target, gene=gene, tau2=float(t2), tau2_estimated=bool(np.isfinite(np.load(Path(model_dir) / 'tau2.npy')[g])),
                   n_backgrounds=int(k), re=float(re), re_se=float(re_se))
    return frame, summary


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)
    f = sub.add_parser('fit')
    for k in ['sources-dir', 'genes', 'out']: f.add_argument('--' + k, required=True)
    f.add_argument('--holdout'); f.add_argument('--sources', nargs='+', default=SOURCES)
    pr = sub.add_parser('predict')
    for k in ['model', 'targets', 'out']: pr.add_argument('--' + k, required=True)
    pr.add_argument('--lines', nargs='+', required=True); pr.add_argument('--chunk', type=int, default=64)
    e = sub.add_parser('exam')
    for k in ['sources-dir', 'holdout', 'out']: e.add_argument('--' + k, required=True)
    e.add_argument('--sources', nargs='+', default=SOURCES)
    x = sub.add_parser('explain')
    for k in ['model', 'target', 'gene']: x.add_argument('--' + k, required=True)
    a = p.parse_args()
    if a.cmd == 'fit': fit(a.sources_dir, a.genes, a.out, a.holdout, a.sources)
    elif a.cmd == 'predict': predict(a.model, a.targets, a.lines, a.out, a.chunk)
    elif a.cmd == 'exam': exam(a.sources_dir, a.holdout, a.out, a.sources)
    else:
        frame, summary = explain(a.model, a.target, a.gene)
        print(frame.to_string(float_format='%.4f')); print(json.dumps(summary, indent=2))

### vcc2_generate.py
Step 6: DE call, α, >5 CPM gate, on-target drop, then cells; also the cell-eval2 exam benchmark.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_generate.py
"""Generate counts with projected composition, metadata cell counts/library sizes,
state-dependent overdispersed induction and reproducible common control sampling.

Finishing: DE call (keep |delta| > z* x SE_pred and |delta| > a minimum size, else 0),
one global alpha, the > 5 CPM gate in the line's own controls, and the knocked-down gene applied separately.

`benchmark` tunes model x z* x minimum size x alpha on the exam (H1-held-out meta-analysis) with official
cell-eval2 (vcc2026), streaming every file one target at a time; the real side is cached once and the ceiling
computed once.
`oracle` is a separate generator ceiling check and never selects submission knobs.
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import shlex
import subprocess
import shutil
import copy
from vcc2_common import int64_indptr, seed_for, validate_counts, save_json, expression_gate, moments, alignment

try:
    from anndata.io import read_elem, write_elem, sparse_dataset
except ImportError:  # older anndata
    from anndata.experimental import read_elem, write_elem, sparse_dataset

VCC2026_METRICS = {
    'pds_cosine': 'max', 'expr_mse_unbiased_capped_norm': 'min',
    'de_wilcoxon_lfc_nmae': 'min', 'de_wilcoxon_direction_fidelity_yield_raw': 'max',
    'de_wilcoxon_direction_reach_raw': 'max', 'de_wilcoxon_sig_jaccard': 'max',
}
CONTROL = "non-targeting"
SCHEMA = "vcc_2026_2_meta_v1"
CC_GENES = ("MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8 HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 "
            "CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 "
            "NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA").split()


class Cfg:
    def __init__(self, a):
        self.alpha = getattr(a, "alpha", 1.0); self.max_lfc = 8.0; self.min_lfc = 0.0
        self.kd_clip = (0.01, 0.9); self.default_kd = 0.30; self.n_cells = getattr(a, "n_cells", 400)
        self.composition = "projection"; self.comp_thr = getattr(a, "comp_thr", 0.02)
        self.pi = getattr(a, "pi", 0.0); self.up_w = getattr(a, "up_w", 0.75)
        self.tilt = getattr(a, "tilt", False); self.match_pool = getattr(a, "match_pool", False) or self.tilt
        self.seed = getattr(a, "seed", 2026)
        self.z = getattr(a, "z", 0.0); self.min_effect = getattr(a, "min_effect", 0.0)   # DE call; 0 and 0 = off
        self.metadata = getattr(a, "metadata", True)
        self.library_size = None
        if (not 0 <= self.pi < 1 or not 0 <= self.up_w <= 1 or self.n_cells < 1 or self.alpha < 0
                or self.z < 0 or self.min_effect < 0):
            raise ValueError("invalid generator settings")

    def as_dict(self):
        return {k: v for k, v in vars(self).items()}


# ---------------------------------------------------------------- inputs
def read_list(path, col):
    return pd.read_csv(path)[col].astype(str).tolist()


def aligned_X(a, genes):
    """cells x our genes, CSR float32 (missing genes = 0, extra genes dropped)."""
    X = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)
    names = a.var_names.astype(str).tolist()
    if names == list(genes):
        return X.astype(np.float32)
    pos = {g: i for i, g in enumerate(genes)}
    src = np.array([i for i, g in enumerate(names) if g in pos]); dst = np.array([pos[names[i]] for i in src])
    X = X[:, src].tocoo()
    print(f"  re-indexed genes: {len(src):,}/{len(genes):,} of ours present")
    return sp.csr_matrix((X.data.astype(np.float32), (X.row, dst[X.col])), shape=(X.shape[0], len(genes)))


class Line:
    """Control cells of one context plus the statistics the generator needs."""
    def __init__(self, X, guide, genes):
        validate_counts(X, "control cells")
        self.X = X.tocsr().astype(np.float32); self.genes = genes; self.n = X.shape[0]
        if not self.n or self.X.sum() <= 0: raise ValueError("controls must contain cells and positive counts")
        self.lib = np.asarray(self.X.sum(1)).ravel()
        self.cpm_cells = sp.diags(1e6 / np.maximum(self.lib.astype(np.float64), 1)) @ self.X
        self.cpm = np.asarray(self.cpm_cells.mean(0)).ravel()           # mean per-cell CPM (scorer gate uses > 5)
        self.share = np.asarray(self.X.sum(0)).ravel() / self.X.sum()   # pooled share of molecules
        self.gate = expression_gate(self.cpm)
        count_mean = np.asarray(self.X.mean(0)).ravel()
        count_var = np.maximum(np.asarray(self.X.multiply(self.X).mean(0)).ravel()-count_mean**2, 0)
        self.overdispersion = np.where(count_mean > 0, np.clip((count_var-count_mean)/np.maximum(count_mean**2, 1e-12), 0, 10), 1.0)
        g = np.asarray(guide).astype(str) if guide is not None else np.zeros(self.n, str)
        self.groups = [np.flatnonzero(g == u) for u in np.unique(g)]
        gp = {x: i for i, x in enumerate(genes)}
        self.cc_idx = np.array([gp[x] for x in CC_GENES if x in gp and self.cpm[gp[x]] > 1], int)
        if len(self.cc_idx) >= 5:
            L = self.cpm_cells[:, self.cc_idx].toarray()
            self.cyc = np.log2(L + 1).mean(1)                            # per-cell cycling level (log2 units)
        else:
            self.cyc = None

    @staticmethod
    def from_h5ad(path, genes, control_col="target_gene"):
        a = ad.read_h5ad(path)
        validate_counts(a.X, str(path))
        if control_col and control_col in a.obs:
            a = a[a.obs[control_col].astype(str).values == CONTROL].copy()
        guide = next((a.obs[c].values for c in ("ntc_id", "guide_id", "sgID_AB", "guide") if c in a.obs), None)
        return Line(aligned_X(a, genes), guide, genes)


def load_deltas(path, targets, genes, array="re", fill=0.0):
    """meta prediction dir (genes.npy, targets.npy, <array>.npy) or .npz (delta [+targets, genes]) -> T x G.

    Missing targets/genes and non-finite values become `fill`: 0 (no change) for deltas, inf (unknown) for SEs.
    """
    p = Path(path)
    if p.is_dir():
        D = np.load(p / f"{array}.npy", mmap_mode="r"); t = np.load(p / "targets.npy").astype(str)
        g = np.load(p / "genes.npy").astype(str)
    else:
        z = np.load(p, allow_pickle=False); D = z["delta"]
        t = z["targets"].astype(str) if "targets" in z else np.array(targets)
        g = z["genes"].astype(str) if "genes" in z else np.array(genes)
    out = np.full((len(targets), len(genes)), fill, np.float32)
    tp = {x: i for i, x in enumerate(t)}; gp = {x: i for i, x in enumerate(g)}
    gsrc = np.array([gp.get(x, -1) for x in genes]); ok = gsrc >= 0
    missing = 0
    for i, x in enumerate(targets):
        if x in tp:
            out[i, ok] = np.asarray(D[tp[x]])[gsrc[ok]]
        else:
            missing += 1
    if missing:
        print(f"  WARNING {path}: {missing} targets missing -> no change predicted for them")
    out[~np.isfinite(out)] = fill
    return out


def load_effects(path, targets, genes, model, cfg):
    """Delta table of one model, plus its SE when the DE call is on."""
    D = load_deltas(path, targets, genes, model)
    se = load_deltas(path, targets, genes, model + "_se", fill=np.inf) if cfg.z > 0 or cfg.min_effect > 0 else None
    return D, se


def kd_from_sources(targets, genes, orion_paths=(), tcell_path=None, tcell_condition="Stim48hr", pseudo=0.5):
    """Knocked-down gene's own log2 drop per target, averaged across sources (M1 rule). NaN if no source has it."""
    gp = {g: i for i, g in enumerate(genes)}; vals = [[] for _ in targets]
    for path in orion_paths:
        with h5py.File(path, "r") as f:
            obs = read_elem(f["obs"]); names = list(obs.index.astype(str)); n = obs["n_cells"].values.astype(float)
            var = read_elem(f["var"]); vg = list(var.index.astype(str))
            vpos = {g: i for i, g in enumerate(vg)}
            c = names.index(CONTROL)
            want = [(i, t) for i, t in enumerate(targets) if t in names and t in vpos]
            if not want:
                continue
            rows = sorted({c} | {names.index(t) for _, t in want})
            S = f["layers/sum_cpm"][rows, :]; rpos = {r: k for k, r in enumerate(rows)}
            ctrl = S[rpos[c]] / n[c]
            for i, t in want:
                r = names.index(t); col = vpos[t]
                vals[i].append(np.log2((S[rpos[r], col] / n[r] + pseudo) / (ctrl[col] + pseudo)))
    if tcell_path:
        t = ad.read_h5ad(tcell_path)
        if "condition" in t.obs:
            t = t[t.obs["condition"].astype(str) == tcell_condition]
        tg = t.obs["target_gene"].astype(str).values; tv = {g: i for i, g in enumerate(t.var_names.astype(str))}
        X = np.asarray(t.X)
        for i, x in enumerate(targets):
            r = np.flatnonzero(tg == x)
            if len(r) and x in tv and np.isfinite(X[r[0], tv[x]]):
                vals[i].append(float(X[r[0], tv[x]]))
    return np.array([np.mean(v) if v else np.nan for v in vals])


def kd_residual(log2_drop, cfg):
    r = np.where(np.isfinite(log2_drop), 2.0 ** np.nan_to_num(log2_drop), cfg.default_kd)
    return np.clip(r, *cfg.kd_clip)


# ---------------------------------------------------------------- core generator
def calibrate(D, line, targets, cfg, se=None):
    """DE call, then the single global alpha and a binary responder-gene expression gate.

    DE call: an effect stays at full size if |delta| > z* x SE_pred and |delta| > min_effect, else 0.
    Unknown SE (inf/NaN) is never convincing. No shrinkage of retained effects.
    The +/-8 generator limit is a numerical safety cap, not an estimated prior.
    On-target suppression is handled separately by measured guide efficiency.
    """
    D = np.asarray(D, np.float32)
    if cfg.z > 0 or cfg.min_effect > 0:
        if se is None: raise ValueError("the DE call needs the prediction SE")
        size = np.abs(D)
        with np.errstate(invalid="ignore"):
            D = np.where((size > cfg.z * np.asarray(se, np.float32)) & (size > cfg.min_effect), D, 0)
    D = np.clip(cfg.alpha * D, -cfg.max_lfc, cfg.max_lfc).astype(np.float32)
    D[:, ~line.gate] = 0.0
    gp = {g: i for i, g in enumerate(line.genes)}
    for i, target in enumerate(targets):
        if target in gp: D[i, gp[target]] = 0.0
    return D


def pick_cells(line, d, cfg, rng):
    """Step 3 (+ optional tilt): which control cells become this knockdown's 400 cells."""
    n = cfg.n_cells
    if cfg.tilt and line.cyc is not None:
        dcc = float(np.mean(d[line.cc_idx]))                       # predicted cycling shift (log2)
        v = line.cyc - line.cyc.mean(); s = max(v.std(), 1e-6)
        if abs(dcc) > 0.02:
            lo, hi = -30.0, 30.0
            for _ in range(50):                                     # bisection: weighted mean shift == dcc
                b = (lo + hi) / 2; w = np.exp(np.clip(b * v / s, -50, 50)); m = (w * v).sum() / w.sum()
                lo, hi = (b, hi) if m < dcc else (lo, b)
            w = np.exp(np.clip(b * v / s, -50, 50)); w /= w.sum()
            k = min(n, int((w > 0).sum()))
            return rng.choice(line.n, k, replace=False, p=w) if k == n else rng.choice(line.n, n, replace=True, p=w)
    if n > line.n:
        order = np.concatenate([rng.permutation(g) for g in line.groups])
        return np.concatenate([order, rng.choice(line.n, n-line.n, replace=True)])
    per = int(np.ceil(n / len(line.groups)))
    pick = np.concatenate([rng.choice(v, size=min(per, len(v)), replace=False) for v in line.groups])
    if len(pick) >= n:
        return rng.choice(pick, n, replace=False)
    rest = np.setdiff1d(np.arange(line.n), pick)
    return np.concatenate([pick, rng.choice(rest, n - len(pick), replace=len(rest) < n - len(pick))])


def project_factors(share, delta, j=None, kd=None):
    """Project all-gene factors onto the composition simplex without an LFC floor.

    The realized expectation is log2(projected_factor), not the unconstrained
    delta. Zero-baseline induction uses the same 0.5-CPM pseudocount as labels.
    """
    fac = np.exp2(np.asarray(delta, float))
    if j is not None and kd is not None: fac[j] = kd
    proposed = np.asarray(share, float)*fac
    zero_induced = (share == 0) & (fac > 1)
    proposed[zero_induced] = 0.5e-6*(fac[zero_induced]-1)
    mass = float(proposed.sum())
    if not np.isfinite(mass) or mass <= 0: raise ValueError("invalid projected composition")
    projected = proposed/mass
    factors = np.divide(projected, share, out=np.ones_like(projected), where=share > 0)
    return factors, projected, mass


def generate_one(line, d, j, kd, cfg, rng=None, pick=None, line_key="line", target_key="target"):
    """Integer counts, with stable per-gene streams shared between perturbations."""
    if rng is None: rng = np.random.default_rng(seed_for(cfg.seed, line_key, target_key))
    if pick is None:
        # Same base pool per line; target order does not advance a line-level stream.
        pick = pick_cells(line, d, cfg, np.random.default_rng(seed_for(cfg.seed, line_key, "base_controls")))
    B = line.X[pick].toarray().astype(np.int64); d = np.asarray(d, float).copy()
    if cfg.match_pool:
        sampled = np.asarray(line.cpm_cells[pick].mean(0)).ravel()
        deviation = np.log2((sampled+0.5)/(line.cpm+0.5))
        mask = line.gate.copy()
        if j is not None: mask[j] = False
        d[mask] = np.clip(d[mask]-deviation[mask], -cfg.max_lfc, cfg.max_lfc)
    factors, projected, mass = project_factors(line.share, d, j, kd)
    # Library size is an independent whole-cell scale after composition projection.
    lib_scale = 1.0
    if cfg.library_size is not None:
        lib_scale = float(cfg.library_size)/max(float(np.median(line.lib[pick])), 1)
    positive = line.share > 0
    effective_pi = min(cfg.pi, float(factors[positive].min())) if positive.any() else 0.0
    esc_rng = np.random.default_rng(seed_for(cfg.seed, line_key, "escapers"))
    escapers = esc_rng.random(len(pick)) < effective_pi
    cols = np.flatnonzero((np.abs(factors*lib_scale-1) > 1e-12) | ((line.share == 0) & (projected > 0)))
    for g in cols:
        # Fresh per-gene seed gives common random draws without target-order dependence.
        grng = np.random.default_rng(seed_for(cfg.seed, line_key, "gene", line.genes[g]))
        factor = factors[g]
        inside = max((factor-effective_pi)/(1-effective_pi), 0)
        F = np.where(escapers, 1.0, inside)*lib_scale
        original = B[:, g].copy()
        values = grng.binomial(original, np.minimum(F, 1.0))
        up = np.maximum(F-1, 0)
        base = line.lib[pick]*line.share[g]
        if line.share[g] == 0 and projected[g] > 0:
            # Induced genes absent from controls must be able to gain counts.
            intensity = line.lib[pick]*projected[g]*lib_scale
        else:
            expected = cfg.up_w*original+(1-cfg.up_w)*base
            intensity = up*expected
        if np.any(intensity > 0):
            dispersion = float(line.overdispersion[g])
            if dispersion > 1e-8:
                intensity = grng.gamma(1/dispersion, intensity*dispersion)
            values += grng.poisson(intensity)
        B[:, g] = values
    if B.max(initial=0) > np.iinfo(np.int32).max: raise ValueError("count exceeds int32 range")
    out = sp.csr_matrix(B.astype(np.int32)); out.eliminate_zeros(); validate_counts(out)
    planned = np.log2((projected*1e6+0.5)/(line.share*1e6+0.5))
    return out, dict(net_share_change=mass-1, projection_normalizer=mass, n_changed=int(len(cols)),
                     effective_pi=effective_pi, library_scale=lib_scale), planned


def target_metadata(path, line, fallback_n, enabled=True):
    """Optional CSV metadata; supports shared or per-context rows/column suffixes."""
    df = pd.read_csv(path)
    if 'target_gene' not in df: raise ValueError("pert_counts.csv needs target_gene")
    if 'context' in df: df = df[df.context.astype(str) == str(line)]
    if df.target_gene.duplicated().any(): raise ValueError("duplicate target rows; supply context labels")
    result = {}
    for row in df.to_dict('records'):
        n, umi = fallback_n, None
        if enabled:
            for field, default in [('n_cells', fallback_n), ('median_umi_per_cell', None)]:
                value = row.get(field+'_'+str(line), row.get(field, default))
                if value is None or pd.isna(value): value = default
                if value is not None and (not np.isfinite(value) or value <= 0): raise ValueError(f"invalid {field} for {row['target_gene']}")
                if field == 'n_cells':
                    if float(value) != int(value): raise ValueError("n_cells must be integer")
                    n = int(value)
                else: umi = None if value is None else float(value)
        result[str(row['target_gene'])] = dict(n_cells=n, median_umi_per_cell=umi)
    if not result: raise ValueError(f"no requested targets for context {line}")
    return result


def check_output(path, expected_shape, block=512):
    a = ad.read_h5ad(path, backed='r')
    try:
        if a.shape != expected_shape: raise ValueError(f"output shape {a.shape} != {expected_shape}")
        stored = 0
        for start in range(0, a.n_obs, block):
            matrix = a.X[start:start+block]; validate_counts(matrix, f"output rows {start}:{start+block}")
            if np.any(np.asarray(matrix.sum(1)).ravel() > 1e6): raise ValueError('cell exceeds 1,000,000 count cap')
            stored += matrix.nnz if sp.issparse(matrix) else matrix.size
        if a.n_obs > 400000 or stored > 4750000000: raise ValueError('submission exceeds cell or stored-entry cap')
    finally: a.file.close()
    return True


def answer_free_check(cells, line, planned, j):
    pooled = np.asarray(cells.sum(0)).ravel(); pooled = pooled/max(pooled.sum(), 1)*1e6
    realized = np.log2((pooled+0.5)/(line.share*1e6+0.5))
    mask = np.isfinite(realized) & (line.share > 0)
    if j is not None: mask[j] = False
    return dict(projected_effect_mae=float(np.abs(realized[mask]-planned[mask]).mean()),
                n_cells=cells.shape[0], median_umi_per_cell=float(np.median(np.asarray(cells.sum(1)).ravel())))


def prediction_ontarget(path, targets):
    p = Path(path)
    if not p.is_dir() or not (p/'ontarget_lfc.npy').exists(): return np.full(len(targets), np.nan)
    names = np.load(p/'targets.npy').astype(str); values = np.load(p/'ontarget_lfc.npy'); pos = dict(zip(names, values))
    return np.asarray([pos.get(t, np.nan) for t in targets])


def cmd_submit(a):
    cfg = Cfg(a)
    if a.settings:
        settings = json.loads(Path(a.settings).read_text())
        if settings.get('schema') != SCHEMA: raise ValueError('use generator settings from this notebook')
        chosen = settings['settings']
        for key in ['alpha', 'z', 'min_effect', 'pi', 'up_w', 'n_cells', 'metadata', 'match_pool', 'tilt']:
            if key in chosen: setattr(cfg, key, chosen[key])
        if a.delta_array == 'selected': a.delta_array = settings['delta_array']
    if a.delta_array == 'selected': raise ValueError('--delta-array selected needs --settings')
    data = Path(a.data); genes = read_list(data/'gene_names.csv', 'gene_name')
    gp = {g: i for i, g in enumerate(genes)}; deltas = dict(x.split('=', 1) for x in a.delta)
    specifications = {line: target_metadata(data/'pert_counts.csv', line, cfg.n_cells, cfg.metadata) for line in a.lines}
    sets = [set(s) for s in specifications.values()]
    if any(s != sets[0] for s in sets): raise ValueError('all submitted contexts must share the exact perturbation panel')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True); path = out/'prediction.h5ad'
    if path.exists(): raise FileExistsError('choose a new output directory to preserve the previous submission')
    for line_specs in specifications.values():
        if any(s['n_cells'] != a.required_n_cells for s in line_specs.values()):
            raise ValueError(f'2026 submissions require exactly {a.required_n_cells} cells per target; optional CSV metadata must respect that contract')
    frames = [pd.DataFrame(dict(target_gene=t, context=line), index=[f'{line}_{t}_{i:05d}' for i in range(spec['n_cells'])])
              for line in a.lines for t, spec in specifications[line].items()]
    obs = pd.concat(frames); obs['target_gene'] = obs.target_gene.astype('category'); obs['context'] = obs.context.astype('category')
    report = dict(settings=cfg.as_dict(), delta_array=a.delta_array, lines={})
    with h5py.File(path, 'w') as f:
        write_elem(f, 'obs', obs); write_elem(f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))
        f.attrs['encoding-type'] = 'anndata'; f.attrs['encoding-version'] = '0.1.0'; ds = None
        for name in a.lines:
            if name not in deltas: raise ValueError(f'no predictions for {name}')
            line = Line.from_h5ad(data/f'context_{name}.h5ad', genes)
            targets = list(specifications[name]); D, se = load_effects(deltas[name], targets, genes, a.delta_array, cfg)
            D = calibrate(D, line, targets, cfg, se); del se
            own = prediction_ontarget(deltas[name], targets)
            if a.kd_orion or a.kd_tcell:
                legacy_own = kd_from_sources(targets, genes, a.kd_orion or (), a.kd_tcell, a.kd_tcell_condition)
                own = np.where(np.isfinite(own), own, legacy_own)
            kd = kd_residual(own, cfg); checks = []; diags = []
            # Pick max count once; different n_cells use prefixes of this same pool.
            pool_cfg = copy.copy(cfg); pool_cfg.n_cells = max(s['n_cells'] for s in specifications[name].values())
            common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(cfg.seed, name, 'base_controls')))
            for i, t in enumerate(targets):
                local = copy.copy(cfg); local.n_cells = specifications[name][t]['n_cells']; local.library_size = specifications[name][t]['median_umi_per_cell']
                pick = None if local.tilt else common[:local.n_cells]
                cells, diag, planned = generate_one(line, D[i], gp.get(t), kd[i], local, pick=pick, line_key=name, target_key=t)
                diags.append(dict(target=t, **diag))
                if i < a.checks: checks.append(dict(target=t, **answer_free_check(cells, line, planned, gp.get(t))))
                # float32 holds these integer counts exactly and matches vcc prep's output encoding,
                # so prep packages without a second full-matrix copy (int32 needs ~37 GB at 360k cells).
                if cells.data.max(initial=0) >= 2**24: raise ValueError('count too large for exact float32')
                cells = cells.astype(np.float32)
                if ds is None: write_elem(f, 'X', int64_indptr(cells)); ds = sparse_dataset(f['X'])
                else: ds.append(cells)
            pd.DataFrame(checks).to_csv(out/f'checks_{name}.csv', index=False)
            pd.DataFrame(diags).to_csv(out/f'projection_{name}.csv', index=False)
            report['lines'][name] = dict(n_targets=len(targets), n_cells=sum(s['n_cells'] for s in specifications[name].values()),
                metadata=specifications[name], measured_ontarget=int(np.isfinite(own).sum()))
            print(f"generated context {name}", flush=True)
    report['integer_counts'] = check_output(path, (len(obs), len(genes)))
    save_json(out/'generator_report.json', report); print(f'WROTE {path}; checked every output row')


class StreamingH5AD:
    """An .h5ad whose obs is known up front and whose X arrives one block at a time.

    Nothing but the current block is held in memory. The row pointer is int64 from the first
    block, so the finished file may exceed 2**31 stored values. Every block is cast to the first
    block's dtype, and the row total is checked against obs on close.
    """
    def __init__(self, path, obs, genes):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        self.n = len(obs); self.rows = 0; self.ds = None; self.dtype = None
        self.f = h5py.File(self.path, 'w')
        self.f.attrs['encoding-type'] = 'anndata'; self.f.attrs['encoding-version'] = '0.1.0'
        write_elem(self.f, 'obs', obs); write_elem(self.f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))

    def append(self, block):
        block = block.tocsr() if sp.issparse(block) else sp.csr_matrix(np.asarray(block))
        if self.ds is None:
            self.dtype = block.dtype
            write_elem(self.f, 'X', int64_indptr(block)); self.ds = sparse_dataset(self.f['X'])
        else:
            self.ds.append(block.astype(self.dtype, copy=False))
        self.rows += block.shape[0]

    def close(self):
        self.f.close()
        if self.rows != self.n: raise ValueError(f'{self.path}: wrote {self.rows} rows for {self.n} obs entries')


def labelled_obs(labels, prefix):
    labels = np.asarray(labels, dtype=str)
    return pd.DataFrame({'target_gene': pd.Categorical(labels)}, index=[f'{prefix}{i:07d}' for i in range(len(labels))])


def stream_h1(a, genes, real_path, line_pseudo=0.5, block=4096):
    """Write the scoring reference straight from the backed H1 file, one target at a time.

    Controls are split into two disjoint, seeded halves: the first half becomes the generator's
    input (held in memory, it is sampled for every target); the second half is appended to the
    reference after the perturbed cells. In oracle mode each target's cells are split the same
    way as before: the estimation half only produces that target's effect vector, the other half
    is appended to the reference. Peak memory is the control half plus one target.

    Returns (Line, targets, {target: reference cell count}, oracle effects or None, oracle own LFCs).
    """
    h = ad.read_h5ad(a.cells, backed='r')
    try:
        labels = h.obs[a.obs_col].astype(str).values
        requested = sorted(set(read_list(a.targets, 'target_gene'))) if a.targets else sorted(set(labels) - {CONTROL})
        counts = pd.Series(labels).value_counts()
        targets = [t for t in requested if counts.get(t, 0) >= a.min_cells and t != CONTROL]
        if a.n_targets and len(targets) > a.n_targets:
            targets = sorted(np.random.default_rng(seed_for(a.seed, 'H1', 'panel')).choice(targets, a.n_targets, replace=False).tolist())
        if not targets: raise ValueError('no eligible H1 targets')

        def subset(rows):
            rows = np.sort(np.asarray(rows))
            small = h[rows].to_memory(); validate_counts(small.X, 'H1 raw counts')
            return aligned_X(small, genes)

        ci = np.flatnonzero(labels == CONTROL)
        if len(ci) < 2: raise ValueError('H1 needs controls')
        ci = np.random.default_rng(seed_for(a.seed, 'H1', 'controls')).permutation(ci)
        train_rows, score_rows = ci[:len(ci) // 2], ci[len(ci) // 2:]
        line = Line(subset(train_rows), None, genes)
        # The scorer uses its REAL control half. Match that exact gate, even when
        # the generator half happens to put a near-threshold gene on the other side.
        score_stats = moments(h, np.sort(score_rows), alignment(h.var_names.astype(str), genes))
        line.gate = expression_gate(score_stats['mean_cpm'])
        save_json(Path(real_path).parent / 'control_split.json', dict(seed=a.seed,
                  generator_controls=len(train_rows), scoring_controls=len(score_rows),
                  gate_source='scoring real controls', genes_above_5_cpm=int(line.gate.sum())))
        plan = {}
        for t in targets:
            rows = np.flatnonzero(labels == t)
            if a.mode == 'oracle':
                rows = np.random.default_rng(seed_for(a.seed, 'H1', t, 'split')).permutation(rows)
                plan[t] = (rows[:len(rows) // 2], rows[len(rows) // 2:])
            else:
                plan[t] = (None, rows)
        real_counts = {t: len(ref) for t, (_, ref) in plan.items()}
        obs_labels = [t for t in targets for _ in range(real_counts[t])] + [CONTROL] * len(score_rows)
        writer = StreamingH5AD(real_path, labelled_obs(obs_labels, 'real_'), genes)
        oracle, own = ([], []) if a.mode == 'oracle' else (None, None)
        gp = {g: i for i, g in enumerate(genes)}
        try:
            for i, t in enumerate(targets):
                estimation, reference = plan[t]
                if estimation is not None:
                    x = subset(estimation); lib = np.asarray(x.sum(1)).ravel()
                    mean = np.asarray((sp.diags(1e6 / np.maximum(lib, 1)) @ x).mean(0)).ravel()
                    d = np.log2((mean + line_pseudo) / (line.cpm + line_pseudo)); oracle.append(d)
                    own.append(d[gp[t]] if t in gp else np.nan); del x
                writer.append(subset(reference))
                print(f'  reference: {i + 1}/{len(targets)} targets', end='\r', flush=True)
            sorted_score = np.sort(score_rows)
            for start in range(0, len(sorted_score), block):
                writer.append(subset(sorted_score[start:start + block]))
        finally:
            writer.close()
        print(f'\n  wrote {real_path}: {writer.rows:,} cells ({len(score_rows):,} scoring controls)', flush=True)
        return (line, targets, real_counts, None if oracle is None else np.asarray(oracle),
                None if own is None else np.asarray(own))
    finally:
        h.file.close()


def write_arm(path, line, effects, kd, targets, n_cells, common, cfg, line_key='H1', block=4096):
    """Generate each target's cells and append them immediately, then the generator's control cells.

    cell-eval2 requires the predicted file to carry the control label even though vcc2026 scores
    every perturbation against the REAL controls (`control_source: real`), so the generator's own
    input controls are appended as before; they are not used as the scoring reference.
    """
    gp = {g: i for i, g in enumerate(line.genes)}
    labels = [t for t in targets for _ in range(n_cells[t])] + [CONTROL] * line.n
    writer = StreamingH5AD(path, labelled_obs(labels, 'pred_'), line.genes)
    try:
        for i, t in enumerate(targets):
            local = copy.copy(cfg); local.n_cells = n_cells[t]
            cells = generate_one(line, effects[i], gp.get(t), kd[i], local, pick=common[:local.n_cells],
                                 line_key=line_key, target_key=t)[0]
            writer.append(cells)
        for start in range(0, line.n, block):
            writer.append(line.X[start:start + block].astype(np.int32))
    finally:
        writer.close()


class OfficialEvaluator:
    """Runs official cell-eval2 against one reference file.

    The reference never changes between arms, so its expensive side (real DE tests,
    pseudobulks, noise corrections) is cached once with `prep-cache --side real` and reused by
    every arm through `--cache-real`. The ceiling depends on the real data only, so it is computed
    once in real-only mode instead of inside every arm.
    """
    def __init__(self, a, reference, out):
        self.binary = shutil.which(a.cell_eval)
        if self.binary is None: raise RuntimeError('install official cell-eval2 (pip install -r requirements-eval.txt) before running the H1 benchmark')
        help_text = subprocess.run([self.binary, 'run', '--help'], check=True, text=True, capture_output=True).stdout
        if '--ceiling' not in help_text or '--cache-real' not in help_text:
            raise RuntimeError('installed cell-eval2 lacks --ceiling/--cache-real; install requirements-eval.txt')
        self.a, self.reference, self.out = a, Path(reference), Path(out)
        self.cache = self.out / 'real_cache'
        self.common = ['--pert-col', 'target_gene', '--control', CONTROL, '--profile', a.eval_profile,
                       '--input-type', 'counts', '--preset', 'vcc2026', '--set', 'de.backend=pdex',
                       '--set', f'num_threads={a.num_threads}']
        self.invocations = []

    def _run(self, command):
        print('  $ ' + ' '.join(shlex.quote(str(x)) for x in command), flush=True)
        subprocess.run([str(x) for x in command], check=True); self.invocations.append([str(x) for x in command])

    def prepare(self):
        # One-sided cache warm: the expression comparator must be named. vcc2026 on counts resolves
        # to bulk_lognorm (cell_eval2.competition.competition_comparator()).
        self._run([self.binary, 'prep-cache', '--side', 'real', '--adata', self.reference,
                   '--cache-real', self.cache, '--comparator', 'bulk_lognorm', *self.common])
        self._run([self.binary, 'run', '-ar', self.reference, '-o', self.out / 'ceiling', '--ceiling',
                   '--cache-real', self.cache, *self.common])

    def score(self, prediction, output):
        self._run([self.binary, 'run', '-ap', prediction, '-ar', self.reference, '-o', output,
                   '--cache-real', self.cache, *self.common])
        self.record(Path(output) / 'invocation.json')

    def record(self, path):
        import importlib.metadata
        versions = {}
        for package in ['cell-eval2', 'pdex', 'numpy', 'scipy', 'anndata']:
            try: versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError: versions[package] = 'not installed in generator environment'
        save_json(path, dict(commands=self.invocations, versions=versions))


def metric_rank_table(outputs, directions_path):
    """Rank only explicitly requested metric families, with explicit directions.

    The evaluator emits diagnostics as well as scoring metrics. Never average
    their ranks blindly. Supply metric,direction ('min'/'max') from current rules.
    """
    directions = pd.read_csv(directions_path)
    if not {'metric', 'direction'} <= set(directions) or directions.metric.duplicated().any():
        raise ValueError('directions CSV needs unique metric,direction columns')
    values = []
    for label, path in outputs.items():
        df = pd.read_csv(Path(path)/'agg_results.csv')
        if not {'metric', 'value'} <= set(df):
            if {'metric', 'score'} <= set(df): df = df.rename(columns={'score': 'value'})
            elif set(directions.metric) <= set(df.columns):
                # cell-eval2 writes a wide describe-style frame; use only its mean row.
                mean_row = np.zeros(len(df), bool)
                for col in df.columns:
                    if col not in set(directions.metric): mean_row |= df[col].astype(str).eq('mean').to_numpy()
                if not mean_row.any() and len(df) == 1: mean_row[:] = True
                if mean_row.sum() != 1: raise ValueError('official aggregate CSV needs exactly one mean row')
                df = pd.DataFrame({'metric': directions.metric, 'value': [df.loc[mean_row, m].iloc[0] for m in directions.metric]})
            else: raise ValueError(f'unrecognized official aggregate schema: {df.columns.tolist()}')
        for metric, direction in directions[['metric', 'direction']].itertuples(index=False, name=None):
            v = df.loc[df.metric == metric, 'value']
            if len(v) != 1 or not np.isfinite(v.iloc[0]): raise ValueError(f'{label}: metric {metric} missing/nonfinite')
            if direction not in ['min', 'max']: raise ValueError('metric direction must be min/max')
            values.append(dict(variant=label, metric=metric, value=float(v.iloc[0]), direction=direction))
    frame = pd.DataFrame(values); frame['rank'] = np.nan
    for metric in directions.metric:
        use = frame.metric == metric
        frame.loc[use, 'rank'] = frame.loc[use, 'value'].rank(ascending=frame.loc[use, 'direction'].iloc[0] == 'min', method='average')
    return frame, frame.groupby('variant')['rank'].mean().sort_values(kind='stable')


def cmd_benchmark(a):
    """Official H1 benchmark, streamed: reference and each arm go to disk one target at a time.

    Oracle mode never writes selected generator settings.
    """
    genes = read_list(a.genes, 'gene_name')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    if a.mode == 'transfer':
        provenance = json.loads((Path(a.predictions)/'prediction_report.json').read_text())
        if provenance.get('heldout_background') != 'H1': raise ValueError('generator tuning requires the exam: predictions with H1 held out')
        if provenance.get('schema') != SCHEMA: raise ValueError('rebuild the exam predictions with vcc2_meta.py')
        if 'H1' in provenance.get('source_backgrounds', []): raise ValueError('H1 leaked into the exam sources')
        unknown = set(a.delta_array) - set(provenance['models'])
        if unknown: raise ValueError(f'unknown models {sorted(unknown)}; the exam has {provenance["models"]}')
    real_path = out/'real.h5ad'
    line, targets, real_counts, oracle_effects, oracle_own = stream_h1(a, genes, real_path)
    evaluator = OfficialEvaluator(a, real_path, out); evaluator.prepare()
    if a.mode == 'transfer':
        kd = kd_residual(prediction_ontarget(a.predictions, targets), Cfg(a))
        effects = {m: (load_deltas(a.predictions, targets, genes, m), load_deltas(a.predictions, targets, genes, m + '_se', fill=np.inf))
                   for m in a.delta_array}
        variants = [dict(alpha=alpha, z=z, min_effect=m, pi=pi, up_w=w, cell_rule=rule)
                    for z in a.z_grid for m in a.min_effect_grid for alpha in a.alpha_grid
                    for pi in a.pi_grid for w in a.up_w_grid for rule in a.cell_rules]
    else:
        # Ceiling-only oracle inputs: per-cell arithmetic CPM from each target's estimation half.
        effects = {'oracle': (oracle_effects, None)}; kd = kd_residual(oracle_own, Cfg(a))
        variants = [dict(alpha=a.alpha, pi=a.pi, up_w=a.up_w, cell_rule='matched')]
    print(f'{len(effects) * len(variants)} arms to score', flush=True)
    outputs, settings = {}, {}
    maximum = max([a.n_cells] + list(real_counts.values()))
    pool_cfg = Cfg(a); pool_cfg.n_cells = maximum
    common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(a.seed, 'H1', 'base_controls')))
    for method, (effects_array, se_array) in effects.items():
        for index, variant in enumerate(variants):
            label = f'{method}_{index:03d}'
            cfg = Cfg(argparse.Namespace(**dict(vars(a), **{k: v for k, v in variant.items() if k != 'cell_rule'})))
            calibrated = calibrate(effects_array, line, targets, cfg, se_array)
            n_cells = {t: (real_counts[t] if variant['cell_rule'] == 'matched' else a.n_cells) for t in targets}
            pred_path = out/f'{label}.h5ad'
            write_arm(pred_path, line, calibrated, kd, targets, n_cells, common, cfg)
            del calibrated
            evaluator.score(pred_path, out/label)
            if not a.keep_arm_files: pred_path.unlink()   # ~2 GB each; the scores are in out/label
            outputs[label] = str(out/label); settings[label] = dict(method=method, **variant)
            save_json(out/f'{label}.json', settings[label])
            print(f'official evaluation complete: {label}', flush=True)
    save_json(out/'benchmark.json', dict(mode=a.mode, outputs=outputs, settings=settings, n_targets=len(targets),
              ceiling=str(out/'ceiling'), real_cache=str(evaluator.cache),
              note='H1 is generator calibration; final evidence requires unseen lines. Oracle is a generator-only ceiling check.'))
    if a.metric_directions or a.eval_profile == 'vcc2026':
        directions = a.metric_directions or str(out/'metric_directions.csv')
        if a.metric_directions is None:
            pd.DataFrame({'metric': list(VCC2026_METRICS), 'direction': list(VCC2026_METRICS.values())}).to_csv(directions, index=False)
        frame, ranks = metric_rank_table(outputs, directions)
        frame.to_csv(out/'metric_ranks.csv', index=False); ranks.to_csv(out/'aggregate_rank.csv')
        if a.mode == 'transfer':
            # Model, z*, minimum size and alpha are chosen together by average rank among the arms run here.
            eligible = [label for label in ranks.index if settings[str(label)]['cell_rule'] == 'fixed']
            if not eligible: raise ValueError('include fixed cell rule to select submission-compatible generator settings')
            best = settings[str(eligible[0])]
            save_json(out/'selected_generator.json', dict(settings=dict({k:v for k,v in best.items() if k not in ['method', 'cell_rule']}, n_cells=a.n_cells),
                       cell_rule='fixed', n_cells=a.n_cells, delta_array=best['method'], arm=str(eligible[0]),
                       selection='official metric average rank on the exam (H1 held out of the meta-analysis)',
                       schema=SCHEMA, note='H1 is calibration, not an untouched test'))


def add_switches(p):
    p.add_argument('--alpha', type=float, default=1.0); p.add_argument('--n-cells', type=int, default=400)
    p.add_argument('--pi', type=float, default=0); p.add_argument('--up-w', type=float, default=0.75)
    p.add_argument('--metadata', action=argparse.BooleanOptionalAction, default=True)
    p.add_argument('--z', type=float, default=0); p.add_argument('--min-effect', type=float, default=0)
    p.add_argument('--match-pool', action='store_true'); p.add_argument('--tilt', action='store_true')
    p.add_argument('--seed', type=int, default=2026)


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__); sub = p.add_subparsers(dest='cmd', required=True)
    s = sub.add_parser('submit'); add_switches(s)
    for key in ['data', 'out']: s.add_argument('--'+key, required=True)
    s.add_argument('--lines', nargs='+', default=['A', 'B', 'C']); s.add_argument('--delta', action='append', required=True)
    s.add_argument('--required-n-cells', type=int, default=400)
    s.add_argument('--delta-array', default='selected'); s.add_argument('--settings')
    s.add_argument('--kd-orion', nargs='*'); s.add_argument('--kd-tcell'); s.add_argument('--kd-tcell-condition', default='Stim48hr')
    s.add_argument('--checks', type=int, default=20)
    for command, mode in [('benchmark', 'transfer'), ('oracle', 'oracle')]:
        b = sub.add_parser(command); add_switches(b); b.set_defaults(mode=mode)
        for key in ['cells', 'genes', 'out']: b.add_argument('--'+key, required=True)
        b.add_argument('--predictions', required=mode == 'transfer'); b.add_argument('--targets')
        b.add_argument('--obs-col', default='target_gene'); b.add_argument('--delta-array', nargs='+', default=['re'])
        b.add_argument('--min-cells', type=int, default=100); b.add_argument('--n-targets', type=int, default=0)
        b.add_argument('--alpha-grid', nargs='+', type=float, default=[0.75, 1.0, 1.5, 2.0])
        b.add_argument('--z-grid', nargs='+', type=float, default=[2.0, 3.0, 4.0])
        b.add_argument('--min-effect-grid', nargs='+', type=float, default=[0.1, 0.2])
        b.add_argument('--pi-grid', nargs='+', type=float, default=[0.0])
        b.add_argument('--up-w-grid', nargs='+', type=float, default=[0.75])
        b.add_argument('--cell-rules', nargs='+', choices=['matched', 'fixed'], default=['fixed'])
        b.add_argument('--cell-eval', default='cell-eval2'); b.add_argument('--eval-profile', default='vcc2026')
        b.add_argument('--num-threads', type=int, default=4)
        b.add_argument('--keep-arm-files', action='store_true', help='keep each scored arm .h5ad (deleted by default)')
        b.add_argument('--metric-directions', help='CSV metric,direction for current official scoring metrics only')
    a = p.parse_args()
    if a.cmd == 'submit': cmd_submit(a)
    else: cmd_benchmark(a)

## 2. Panel sources
Rebuild `lib/sources` (QC-filtered knockdowns on the shared panel) only if it is missing, then show how many backgrounds cover each target.

In [ ]:
VOTERS = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr', 'H1']   # step 1: CD4_Rest / CD4_Stim8hr do not vote
panels = [Path(ENV['SOURCES']) / s / 'effects_panel.h5ad' for s in VOTERS]
if not all(p.exists() for p in panels):
    bg('panels', r"""
    "$PY" "$SCRIPTS/knockdown_filters.py"
    "$PY" "$SCRIPTS/build_panel_sources.py"
    """)
else:
    cov = pd.read_csv(Path(ENV['SOURCES']) / 'coverage.csv', index_col='target')
    kept = cov[VOTERS].eq('kept')
    print('panel targets by number of voting backgrounds (ours = the 300 challenge targets):')
    display(pd.crosstab(kept.sum(axis=1).rename('backgrounds'), cov['panel']))

## 3. Exam
Fit τ² without H1, list H1 targets that another background also kept, and predict them with both models.

In [ ]:
run(r"""
"$PY" vcc2_meta.py fit --sources-dir "$SOURCES" --genes "$DATA/gene_names.csv" --holdout H1 --out "$WORK/models/exam"
"$PY" vcc2_meta.py exam --sources-dir "$SOURCES" --holdout H1 --out "$WORK/exam/targets.csv"
"$PY" vcc2_meta.py predict --model "$WORK/models/exam" --targets "$WORK/exam/targets.csv" --lines H1 --out "$WORK/exam/pred"
""")

Worked example from the exam model: each background's lfc, SE, weight and share for one target × gene (gene = its strongest effect), plus τ² across genes.

In [ ]:
sys.path.insert(0, ENV['CODE']); import vcc2_meta
EXAMPLE_TARGET, EXAMPLE_GENE = 'ACLY', None
exam_dir = Path(ENV['WORK']) / 'exam/pred/H1'
if EXAMPLE_GENE is None:
    row = list(np.load(exam_dir / 'targets.npy')).index(EXAMPLE_TARGET)
    z = np.abs(np.load(exam_dir / 're.npy', mmap_mode='r')[row]) / np.load(exam_dir / 're_se.npy', mmap_mode='r')[row]
    EXAMPLE_GENE = str(np.load(exam_dir / 'genes.npy')[np.nanargmax(np.where(np.load(exam_dir / 'n_backgrounds.npy')[row] >= 2, z, np.nan))])
frame, summary = vcc2_meta.explain(Path(ENV['WORK']) / 'models/exam', EXAMPLE_TARGET, EXAMPLE_GENE)
display(frame.round(4)); print(json.dumps(summary, indent=1))
print('tau2 quantiles (exam):', json.loads((Path(ENV['WORK']) / 'models/exam/model.json').read_text())['tau2_quantiles'])

## 4. Final model
Fit τ² on all five backgrounds and write one Δ table per challenge line.

In [ ]:
run(f"""
"$PY" vcc2_meta.py fit --sources-dir "$SOURCES" --genes "$DATA/gene_names.csv" --out "$WORK/models/final"
"$PY" vcc2_meta.py predict --model "$WORK/models/final" --targets "$DATA/pert_counts.csv" --lines {' '.join(LINES)} --out "$WORK/pred"
""")

### Future: context-aware weighting (not in this submission)
The Δ tables above are identical for A, B and C: nothing about the hidden line enters the average. Lines differ only in the generator, which uses each line's own controls (baseline, dispersion, library sizes, >5 CPM gate).
Letting the line's profile change the average itself, e.g. weighting backgrounds by how similar their controls are to line A, would be a new feature and is not part of the current spec. Plan: profile the context of each line entering the model and weight changes towards the backgrounds it most resembles, once this submission has set the baseline.

**Where it goes:** the averaging step (`combine` in `vcc2_meta.py`), not the cell generator. The generator receives a finished Δ per line; by then the background weights are used up.

**Weights:** multiply a similarity score s(line, background) into the current weight:

- now: w = 1 / (SE² + τ²)
- with similarity: w = s(line, background) / (SE² + τ²)

**SE must change with it** (it drives the DE call). √(1/Σw) only holds when w is exactly 1/(SE² + τ²). For any other weights:

SE_pred = √( Σ w² · (SE² + τ²) ) / Σ w

(With s = 1 for every background this reduces to √(1/Σw), so the current results are unchanged.)

**What stays the same:**
- τ² measures how much the sources disagree with each other, independent of the line being predicted, so `fit` is unchanged.
- `predict` already writes one folder per line; with similarity the A/B/C tables simply differ, and the generator needs no change.

**Test it on the exam first:** score H1's similarity to the four sources, rerun section 3 and the exam benchmark, and compare against this submission's baseline before using it for A/B/C.

**The hard part** is the profile itself: a similarity score that is comparable across sources measured on different platforms (Orion pseudobulk, Replogle K562, T-cell pseudobulk, H1 Flex).

## 5. Tune on the exam
Stage A: z* {2, 3, 4} × min size {0.1, 0.2} at α = 1 (6 arms, ~9 min each on 25 targets).

In [ ]:
BENCHMARK_TARGETS = 25
EXAM = (f'--cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK/exam/targets.csv" --predictions "$WORK/exam/pred/H1" '
        f'--eval-profile vcc2026 --cell-rules fixed --n-targets {BENCHMARK_TARGETS} --pi-grid 0 --up-w-grid 0.75')
#bg('exam_a', f'"$PY" vcc2_generate.py benchmark {EXAM} --delta-array re --z-grid 2 3 4 --min-effect-grid 0.1 0.2 --alpha-grid 1 --out "$WORK/exam_a"')

Stage A progress, then every arm's settings with its average rank.

In [ ]:
def arms(folder):
    folder = Path(ENV['WORK']) / folder
    ranks = pd.read_csv(folder / 'aggregate_rank.csv', index_col=0)
    table = pd.DataFrame({label: json.loads((folder / f'{label}.json').read_text()) for label in ranks.index}).T
    values = pd.read_csv(folder / 'metric_ranks.csv').pivot(index='variant', columns='metric', values='value')
    return table.join(values).join(ranks).sort_values('rank')
status('exam_a', n=5)
if (Path(ENV['WORK']) / 'exam_a/selected_generator.json').exists(): display(arms('exam_a'))

Stage B: the stage-A winner's z* and min size × α {0.75, 1, 1.5, 2}.

In [ ]:
best = json.loads((Path(ENV['WORK']) / 'exam_a/selected_generator.json').read_text())
model, s = best['delta_array'], best['settings']
print(f"stage A winner: z*={s['z']}, min size={s['min_effect']}")
bg('exam_b', f'"$PY" vcc2_generate.py benchmark {EXAM} --delta-array {model} --z-grid {s["z"]} --min-effect-grid {s["min_effect"]} --alpha-grid 0.75 1 1.5 2 --out "$WORK/exam_b"')

Stage B progress, arms and the selected settings (these drive generation).

In [ ]:
status('exam_b', n=5)
settings = Path(ENV['WORK']) / 'exam_b/selected_generator.json'
if settings.exists():
    display(arms('exam_b')); print(settings.read_text())

Optional: generator ceiling on H1 using effects estimated from half of each target's own cells; never selects settings.

In [ ]:
RUN_ORACLE = False
if RUN_ORACLE:
    bg('oracle_h1', f'"$PY" vcc2_generate.py oracle --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK/exam/targets.csv" --eval-profile vcc2026 --n-targets {BENCHMARK_TARGETS} --out "$WORK/oracle_h1"')

## 6. Generate
Cells for every challenge line from its Δ table with the stage-B settings (z*, min size, α); refuses to overwrite a finished run.

In [ ]:
settings = Path(ENV['WORK']) / 'exam_b/selected_generator.json'
if not settings.exists(): raise FileNotFoundError('Finish exam stage B before generation.')
result = V / 'results' / RUN_NAME
if result.exists() and any(result.iterdir()): raise FileExistsError('Choose a new RUN_NAME to preserve existing results.')
line_args = ' '.join(shlex.quote(line) for line in LINES)
delta_args = ' '.join(f'--delta {shlex.quote(line + "=" + str(Path(ENV["WORK"]) / "pred" / line))}' for line in LINES)
bg('generate', f'"$PY" vcc2_generate.py submit --data "$DATA" --lines {line_args} --delta-array re --settings {shlex.quote(str(settings))} {delta_args} --out {shlex.quote(str(result))}')

In [ ]:
status('generate')

## 7. Package
`vcc prep` validates and packages the run; leaderboard submission and bucket backup stay off until switched on.

In [ ]:
SUBMISSION = V / 'results' / RUN_NAME
run(f'vcc prep {shlex.quote(str(SUBMISSION / "prediction.h5ad"))} -g "$DATA/gene_names.csv" --perts "$DATA/pert_counts.csv" -o {shlex.quote(str(SUBMISSION / "prediction.vcc"))}')

In [ ]:
SUBMIT_TO_LEADERBOARD = True
MODEL_NAME = 'LHY'   # shown publicly on the leaderboard: keep it free of method details
if SUBMIT_TO_LEADERBOARD:
    chosen = json.loads((Path(ENV['WORK']) / 'exam_b/selected_generator.json').read_text())
    s = chosen['settings']
    print(f"submitting as {MODEL_NAME!r}; private record: {chosen['delta_array']}, z*={s['z']}, min={s['min_effect']}, alpha={s['alpha']}")
    run(f'vcc submit {shlex.quote(str(SUBMISSION / "prediction.vcc"))} -m {shlex.quote(MODEL_NAME)} --wait')


In [ ]:
BACK_UP_TO_BUCKET = True
if BACK_UP_TO_BUCKET:
    run(r"""
    gcloud storage rsync -r "$WORK" "$B/lib/vcc_2026_2" --exclude='.*\.h5ad$'
    gcloud storage rsync -r "$CODE" "$B/code/vcc_2026_2"
    gcloud storage cp ~/vcc/vcc_2026_2.ipynb "$B/code/vcc_2026_2.ipynb"
    """)

## Ad hoc tests
Diagnostics from the post-submission review (step 0). Nothing here changes the submission or the exam results; everything is written to `lib/vcc_2026_2/adhoc/`.

Write the diagnostics script.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_adhoc.py
"""Ad hoc diagnostics after the vcc_2026_2 submission (step 0 of the review). Nothing here changes the submission.

  arms  regenerate four arms on the exam's 25 H1 targets, reusing the exam's real file and real-side cache, and score
        each with cell-eval2, keeping both sides' DE tables (--write-degenes):
          null_shared       delta = 0 and no knockdown; every target is the one shared 400-cell pool (pure pool artefact)
          null_independent  delta = 0 and no knockdown; a fresh 400-cell draw per target
          submitted         the submitted settings (exam_b's selection)
          z2                z* = 2, min 0.1, alpha 1 (stage A's best arm on fid and Jaccard)
        The shared-pool arms rebuild the benchmark's cells exactly (same control split, pool and seeds), so 'submitted'
        must score as its exam_b arm did.

Functions used by the notebook cells:
  de_table(arm)        per-target predicted vs real DE genes (scorer's p_adj < 0.05, own gene excluded)
  generic_share(...)   how much of H1's real DE lands on genes DE in > 30% of knockdowns
  generic_transfer()   correlation of each source's average response with H1's
"""
from __future__ import annotations

import argparse
import json
import subprocess
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
from scipy import stats

import vcc2_generate as g
from vcc2_common import alignment, expression_gate, moments, seed_for, save_json

try:
    from anndata.io import read_elem
except ImportError:  # older anndata
    from anndata.experimental import read_elem

CONTROL = 'non-targeting'
SEED = 2026
ARMS = ['null_shared', 'null_independent', 'submitted', 'z2']
SOURCES = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr']


# ---------------------------------------------------------------- arms
def exam_line(cells, genes, seed=SEED):
    """The benchmark's generator controls and its scoring-half gate, rebuilt as stream_h1 does."""
    h = ad.read_h5ad(cells, backed='r')
    try:
        labels = h.obs['target_gene'].astype(str).values
        ci = np.random.default_rng(seed_for(seed, 'H1', 'controls')).permutation(np.flatnonzero(labels == CONTROL))
        train, score = ci[:len(ci) // 2], ci[len(ci) // 2:]
        small = h[np.sort(train)].to_memory()
        line = g.Line(g.aligned_X(small, genes), None, genes); del small
        line.gate = expression_gate(moments(h, np.sort(score), alignment(h.var_names.astype(str), genes))['mean_cpm'])
    finally:
        h.file.close()
    return line


def exam_targets(real):
    a = ad.read_h5ad(real, backed='r')
    try: counts = a.obs['target_gene'].astype(str).value_counts()
    finally: a.file.close()
    counts = counts.drop(CONTROL)
    return sorted(counts.index), counts.to_dict()


def write_arm(path, line, targets, effects, kd, picks, cfg, own):
    labels = [t for t in targets for _ in range(cfg.n_cells)] + [CONTROL] * line.n
    writer = g.StreamingH5AD(path, g.labelled_obs(labels, 'pred_'), line.genes)
    try:
        for i, t in enumerate(targets):
            writer.append(g.generate_one(line, effects[i], own[i], None if kd is None else kd[i], cfg,
                                         pick=picks[i], line_key='H1', target_key=t)[0])
        for start in range(0, line.n, 4096):
            writer.append(line.X[start:start + 4096].astype(np.int32))
    finally:
        writer.close()


def score(binary, pred, real, cache, out, threads=4):
    command = [binary, 'run', '-ap', pred, '-ar', real, '-o', out, '--cache-real', cache, '--pert-col', 'target_gene',
               '--control', CONTROL, '--profile', 'vcc2026', '--input-type', 'counts', '--preset', 'vcc2026',
               '--set', 'de.backend=pdex', '--set', f'num_threads={threads}', '--write-degenes']
    print('  $ ' + ' '.join(map(str, command)), flush=True)
    subprocess.run([str(x) for x in command], check=True)


def cmd_arms(a):
    genes = g.read_list(a.genes, 'gene_name'); exam = Path(a.exam); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    targets, real_counts = exam_targets(exam / 'real.h5ad')
    print(f'{len(targets)} exam targets; rebuilding the generator controls from {a.cells}', flush=True)
    line = exam_line(a.cells, genes)
    selected = json.loads((exam / 'selected_generator.json').read_text())['settings']
    settings = dict(submitted=dict(alpha=selected['alpha'], z=selected['z'], min_effect=selected['min_effect']),
                    z2=dict(alpha=1.0, z=2.0, min_effect=0.1))
    base = dict(pi=0.0, up_w=0.75, n_cells=a.n_cells, seed=SEED)
    pool_cfg = g.Cfg(argparse.Namespace(**base)); pool_cfg.n_cells = max([a.n_cells] + list(real_counts.values()))
    common = g.pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(SEED, 'H1', 'base_controls')))
    shared = [common[:a.n_cells]] * len(targets)
    cfg0 = g.Cfg(argparse.Namespace(**base))
    independent = [g.pick_cells(line, np.zeros(len(genes)), cfg0, np.random.default_rng(seed_for(SEED, 'H1', 'independent', t)))
                   for t in targets]
    gp = {x: i for i, x in enumerate(genes)}; own = [gp.get(t) for t in targets]
    zeros = np.zeros((len(targets), len(genes)), np.float32)
    D = g.load_deltas(a.predictions, targets, genes, 're'); se = g.load_deltas(a.predictions, targets, genes, 're_se', fill=np.inf)
    plan = dict(null_shared=(zeros, None, shared, [None] * len(targets), cfg0),
                null_independent=(zeros, None, independent, [None] * len(targets), cfg0))
    for name, s in settings.items():
        cfg = g.Cfg(argparse.Namespace(**base, **s))
        plan[name] = (g.calibrate(D, line, targets, cfg, se), g.kd_residual(g.prediction_ontarget(a.predictions, targets), cfg),
                      shared, own, cfg)
    for name in a.arms:
        effects, kd, picks, owns, cfg = plan[name]
        path = out / f'{name}.h5ad'
        write_arm(path, line, targets, effects, kd, picks, cfg, owns)
        score(a.cell_eval, path, exam / 'real.h5ad', exam / 'real_cache', out / name, a.num_threads)
        path.unlink()                                             # ~2 GB; the scores and DE tables are kept
        save_json(out / name / 'arm.json', dict(arm=name, settings={k: v for k, v in vars(cfg).items() if k in ['alpha', 'z', 'min_effect']},
                                                pool='independent' if name == 'null_independent' else 'shared', targets=targets))
        print(f'arm complete: {name}', flush=True)


# ---------------------------------------------------------------- analysis
def de_table(arm_dir, alpha=0.05):
    """Per target: predicted and real DE genes (p_adj < alpha, own gene excluded), overlap, precision, recall, and the
    chance levels: random genes would hit real DE genes at n_real / scored genes, and get the direction right half the time."""
    pred = pd.read_parquet(Path(arm_dir) / 'de_pred.parquet'); real = pd.read_parquet(Path(arm_dir) / 'de_real.parquet')
    rows = []
    for t, r in real.groupby('target'):
        p = pred[pred.target == t]
        P = set(p.feature[(p.p_adj < alpha) & (p.feature != t)]); R = set(r.feature[(r.p_adj < alpha) & (r.feature != t)])
        both = P & R
        same_sign = 0
        if both:
            lp = p.set_index('feature').log2_fold_change; lr = r.set_index('feature').log2_fold_change
            same_sign = int(sum(np.sign(lp[x]) == np.sign(lr[x]) for x in both))
        scored = r.feature.nunique()
        precision = len(both) / len(P) if P else np.nan
        rows.append(dict(target=t, n_pred=len(P), n_real=len(R), overlap=len(both), same_direction=same_sign,
                         precision=precision, chance_precision=len(R) / scored, enrichment=precision * scored / len(R) if R else np.nan,
                         direction_agreement=same_sign / len(both) if both else np.nan, recall=len(both) / len(R) if R else np.nan))
    return pd.DataFrame(rows).set_index('target')


def generic_from_de(de, alpha=0.05, share=0.3):
    """From a scorer DE table: genes DE in > `share` of targets, and the fraction of all DE calls on them."""
    calls = de[(de.p_adj < alpha) & (de.feature != de.target)]
    n_targets = de.target.nunique()
    freq = calls.groupby('feature').target.nunique() / n_targets
    generic = set(freq.index[freq > share])
    return dict(targets=n_targets, de_calls=len(calls), generic_genes=len(generic),
                calls_on_generic=int(calls.feature.isin(generic).sum()),
                share_of_calls_on_generic=float(calls.feature.isin(generic).mean()) if len(calls) else np.nan), generic, freq


def effect_table(path, min_cpm):
    with h5py.File(path, 'r') as f:
        obs, var = read_elem(f['obs']), read_elem(f['var'])
        keep = np.flatnonzero(var['ctrl_cpm'].to_numpy(float) > min_cpm)
        lfc = f['X'][:][:, keep]; se = f['layers/se'][:][:, keep]
    genes = var.index.astype(str).to_numpy()[keep]; targets = obs.index.astype(str).to_numpy()
    gp = {x: i for i, x in enumerate(genes)}
    for i, t in enumerate(targets):                                   # own gene is not a downstream response
        if t in gp: lfc[i, gp[t]] = np.nan
    return targets, genes, lfc, se


def generic_from_effects(path, alpha=0.05, share=0.3, min_cpm=5.0):
    """Same question on a pseudobulk effect table: z = lfc / SE, BH within each knockdown (like the scorer's per-pert FDR)."""
    targets, genes, lfc, se = effect_table(path, min_cpm)
    with np.errstate(invalid='ignore', divide='ignore'):
        p = 2 * stats.norm.sf(np.abs(lfc / se))
    de = np.zeros_like(lfc, bool)
    for i in range(len(targets)):
        ok = np.isfinite(p[i])
        if ok.any(): de[i, np.flatnonzero(ok)] = stats.false_discovery_control(p[i, ok]) < alpha
    freq = de.mean(0); generic = freq > share
    calls = int(de.sum())
    return dict(targets=len(targets), genes=len(genes), de_calls=calls, generic_genes=int(generic.sum()),
                calls_on_generic=int(de[:, generic].sum()),
                share_of_calls_on_generic=float(de[:, generic].sum() / calls) if calls else np.nan), pd.Series(freq, index=genes)


def average_response(path, min_cpm=1.0):
    targets, genes, lfc, _ = effect_table(path, min_cpm)
    return pd.Series(np.nanmean(lfc, 0), index=genes), len(targets)


def generic_transfer(filters_dir, h1_generic_freq, real_de=None, top=200):
    """Each source's average knockdown response vs H1's: Pearson r, and overlap of the strongest genes."""
    h1, n_h1 = average_response(Path(filters_dir) / 'H1.h5ad', min_cpm=5.0)
    references = {'H1 effect table (all kept knockdowns)': h1}
    if real_de is not None:
        references['H1 scorer DE (25 exam targets)'] = real_de[real_de.feature != real_de.target].groupby('feature').log2_fold_change.mean()
    sources = {name: average_response(Path(filters_dir) / f'{name}.h5ad')[0] for name in SOURCES}
    sources['mean of the 4 sources'] = pd.concat(sources.values(), axis=1).mean(axis=1)
    generic = set(h1_generic_freq.index[h1_generic_freq > 0.3])
    rows = []
    for ref_name, ref in references.items():
        for name, s in sources.items():
            common = ref.index.intersection(s.index); x, y = s[common], ref[common]
            ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
            top_ref = set(y.abs().nlargest(top).index); top_src = set(x.abs().nlargest(top).index)
            src_on_generic = set(x.abs().nlargest(len(generic)).index) & generic if generic else set()
            rows.append(dict(reference=ref_name, source=name, genes=len(x), pearson=float(np.corrcoef(x, y)[0, 1]),
                             spearman=float(stats.spearmanr(x, y).statistic), **{f'top{top}_overlap': len(top_ref & top_src) / top},
                             same_sign_on_top=float(np.mean(np.sign(x[list(top_ref & top_src)]) == np.sign(y[list(top_ref & top_src)])))
                             if top_ref & top_src else np.nan,
                             h1_generic_recovered=len(src_on_generic) / len(generic) if generic else np.nan,
                             source_mean_abs=float(x.abs().mean()), h1_mean_abs=float(y.abs().mean())))
    return pd.DataFrame(rows).set_index(['reference', 'source'])


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)
    s = sub.add_parser('arms')
    for key in ['cells', 'genes', 'exam', 'predictions', 'out']: s.add_argument('--' + key, required=True)
    s.add_argument('--arms', nargs='+', default=ARMS, choices=ARMS); s.add_argument('--n-cells', type=int, default=400)
    s.add_argument('--cell-eval', default='cell-eval2'); s.add_argument('--num-threads', type=int, default=4)
    a = p.parse_args()
    cmd_arms(a)

Regenerate four arms on the exam's 25 H1 targets and score them with cell-eval2, keeping both DE tables (background, ~1 h): no change from the shared pool, no change from an independent pool per knockdown, the submitted settings, and z* = 2.

In [ ]:
ADHOC = Path(ENV['WORK']) / 'adhoc'
bg('adhoc_arms', f'"$PY" vcc2_adhoc.py arms --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --exam "$WORK/exam_b" --predictions "$WORK/exam/pred/H1" --out {shlex.quote(str(ADHOC))}')

In [ ]:
status('adhoc_arms', n=5)

Check: the regenerated submitted arm must score exactly as its exam_b arm did (same cells, same reference).

In [ ]:
sys.path.insert(0, ENV['CODE']); import vcc2_adhoc
ADHOC = Path(ENV['WORK']) / 'adhoc'
def agg_means(folder):
    frame = pd.read_csv(Path(folder) / 'agg_results.csv')
    return frame[frame.statistic == 'mean'].iloc[0].drop('statistic').astype(float)
chosen = json.loads((Path(ENV['WORK']) / 'exam_b/selected_generator.json').read_text())['arm']
if (ADHOC / 'submitted/agg_results.csv').exists():
    check = pd.DataFrame({'exam_b ' + chosen: agg_means(Path(ENV['WORK']) / 'exam_b' / chosen), 'regenerated': agg_means(ADHOC / 'submitted')})
    display(check.round(5)); print('identical:', bool(np.allclose(check.iloc[:, 0], check.iloc[:, 1], equal_nan=True)))
else:
    print('waiting for adhoc_arms')

D1 — pool artefact: DE genes per knockdown when nothing is predicted. Anything above ~0 comes from the pool, not the model.

In [ ]:
if (ADHOC / 'null_independent/de_pred.parquet').exists():
    d1 = pd.DataFrame({arm: vcc2_adhoc.de_table(ADHOC / arm)['n_pred'] for arm in ['null_shared', 'null_independent']})
    display(d1.describe().round(1).T)
    shared = pd.read_parquet(ADHOC / 'null_shared/de_pred.parquet')
    hits = shared[(shared.p_adj < 0.05) & (shared.feature != shared.target)].feature.value_counts()
    print(f'shared pool: {int((hits == 25).sum())} genes called DE in all 25 knockdowns, {len(hits)} in at least one')
else:
    print('waiting for adhoc_arms')

D2 — our predicted DE genes vs the real ones per knockdown (scorer's p_adj < 0.05), for the submitted and z* = 2 arms. Precision is compared with chance (random genes hit real DE genes at n_real / scored genes) and direction agreement with 50%.

In [ ]:
if (ADHOC / 'z2/de_pred.parquet').exists():
    d2 = {arm: vcc2_adhoc.de_table(ADHOC / arm) for arm in ['submitted', 'z2']}
    summary = pd.DataFrame({arm: dict(median_n_pred=t.n_pred.median(), median_n_real=t.n_real.median(),
                                      median_precision=t.precision.median(), median_chance_precision=t.chance_precision.median(),
                                      median_enrichment=t.enrichment.median(), targets_above_chance=int((t.enrichment > 1).sum()),
                                      direction_agreement=t.same_direction.sum() / max(t.overlap.sum(), 1),
                                      direction_agreement_without_TADA1=t.drop('TADA1').same_direction.sum() / max(t.drop('TADA1').overlap.sum(), 1),
                                      median_recall=t.recall.median())
                            for arm, t in d2.items()}).T
    display(summary.round(3))
    cols = ['n_pred', 'n_real', 'overlap', 'precision', 'chance_precision', 'enrichment', 'direction_agreement', 'recall']
    display(d2['submitted'][cols].join(d2['z2'][cols], rsuffix='_z2').sort_values('n_real', ascending=False).round(3))
else:
    print('waiting for adhoc_arms')

D3 — how much of H1's real DE is generic (DE in > 30% of knockdowns): from the scorer's DE on the 25 exam targets, and from the H1 effect table across all kept knockdowns (|lfc| / SE test, BH within each knockdown, genes > 5 CPM).

In [ ]:
sys.path.insert(0, ENV['CODE']); import vcc2_adhoc
real_de = pd.read_parquet(next((Path(ENV['WORK']) / 'exam_b/real_cache').glob('de_wilcoxon_table-*.parquet')))
scorer, scorer_generic, scorer_freq = vcc2_adhoc.generic_from_de(real_de)
table, h1_freq = vcc2_adhoc.generic_from_effects(Path(ENV['LIB']) / 'filters/H1.h5ad')
display(pd.DataFrame({'scorer DE, 25 exam targets': scorer, 'H1 effect table, all kept knockdowns': table}).T)
print('most frequently DE genes (scorer, share of the 25 targets):')
print(scorer_freq.sort_values(ascending=False).head(15).round(2).to_string())

D4 — does the sources' average knockdown response predict H1's? Pearson / Spearman r, overlap of the 200 strongest genes, and the share of H1's generic genes (D3) each source recovers.

In [ ]:
d4 = vcc2_adhoc.generic_transfer(Path(ENV['LIB']) / 'filters', h1_freq, real_de)
display(d4.round(3))